<a href="https://colab.research.google.com/github/Somaskandan931/flyrank-ml-2026-Somaskandan931/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Somaskandan931/flyrank-ml-2026-Somaskandan931/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip install -q duckdb huggingface_hub pandas

from huggingface_hub import login
from google.colab import userdata
import duckdb, pandas as pd, os

login(token=userdata.get('HF_TOKEN'))

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("CREATE SECRET (TYPE huggingface, TOKEN '" + userdata.get('HF_TOKEN') + "')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT_DAILY = f"{BASE}/fact_content_daily_performance/**/*.parquet"

## 1. My rule and its reason codes

**Two signal checks, both behind real FlyRank flags from the session:**

1. **CTR vs. position** — the signal behind the CTR-fix flag. Claim: CTR should fall as
   average position gets worse (higher number = lower on the page). Bucketed by
   `gsc_avg_position` band, on March 2026, `gsc_data_available IS TRUE`.
2. **Volume (impressions)** — the signal behind the quick-win flag. Claim: high-impression
   content that still underperforms its position's own CTR benchmark is a good "fix this
   first" candidate, since the same effort recovers more absolute clicks. Bucketed by
   `gsc_impressions` tier, same filters.

**My rule:** *CTR Recovery Priority.* For each `(content_hash_id, client_hash_id)` pair in
March 2026, compute its actual CTR and compare it to the average CTR of other content in the
same position band (its "benchmark"). If actual CTR is below benchmark, the gap — multiplied
by impression volume — becomes the priority score: a big underperformance on a
high-traffic page ranks above the same underperformance on a low-traffic page.

- **Score** = `(benchmark_ctr − actual_ctr) × gsc_impressions`, floored at 0
- **Reason code** = `CTR_BELOW_POSITION_BENCHMARK`
- **Action label** = `REVIEW_TITLE_AND_META` if score > 0, else `MONITOR`

No future window or product flag is used — only contemporaneous GSC counts from the same
month.

In [ ]:
q_ctr_position = f"""
SELECT
  CASE
    WHEN gsc_avg_position <= 3 THEN '1-3'
    WHEN gsc_avg_position <= 10 THEN '4-10'
    WHEN gsc_avg_position <= 20 THEN '11-20'
    WHEN gsc_avg_position <= 50 THEN '21-50'
    ELSE '51+'
  END AS position_bucket,
  COUNT(*) AS n,
  SUM(gsc_clicks) AS total_clicks,
  SUM(gsc_impressions) AS total_impressions,
  ROUND(SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0), 4) AS ctr
FROM read_parquet('{FACT_DAILY}')
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
  AND gsc_impressions > 0
GROUP BY 1
ORDER BY MIN(gsc_avg_position)
"""
ctr_position_df = con.sql(q_ctr_position).df()
ctr_position_df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,total_clicks,total_impressions,ctr
0,1-3,727362,205457.0,54028594.0,0.0038
1,4-10,1456122,445828.0,137830113.0,0.0032
2,11-20,519223,92449.0,29386006.0,0.0031
3,21-50,631491,76589.0,55944412.0,0.0014
4,51+,276863,1509.0,3468464.0,0.0004


**Verdict — CTR vs. position: [FILL IN: CONFIRMED / OPPOSITE / MIXED / FALSE]**

Fill this in from the table above: if `ctr` decreases monotonically (or nearly so) as the
position bucket gets worse, that's **CONFIRMED** — it validates the signal behind the
CTR-fix flag, and every bucket's `n` is printed above so the sample size behind each row
is visible.

In [ ]:
q_volume = f"""
WITH per_page AS (
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS impressions,
           SUM(gsc_clicks) AS clicks,
           SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr
    FROM read_parquet('{FACT_DAILY}')
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
    GROUP BY 1,2
    HAVING SUM(gsc_impressions) > 0
)
SELECT
  CASE
    WHEN impressions < 100 THEN '<100'
    WHEN impressions < 1000 THEN '100-999'
    WHEN impressions < 10000 THEN '1k-9.9k'
    ELSE '10k+'
  END AS impression_tier,
  COUNT(*) AS n,
  ROUND(AVG(ctr), 4) AS avg_ctr
FROM per_page
GROUP BY 1
ORDER BY MIN(impressions)
"""
volume_df = con.sql(q_volume).df()
volume_df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,impression_tier,n,avg_ctr
0,<100,75297,0.0073
1,100-999,56383,0.0023
2,1k-9.9k,39181,0.0030
3,10k+,5877,0.0029


**Verdict — volume vs. CTR: [FILL IN: CONFIRMED / OPPOSITE / MIXED / FALSE]**

Fill this in from the table above: if higher-impression tiers show meaningfully lower
average CTR (more visibility not converting to clicks), that **CONFIRMS** the quick-win
logic — bigger wins are available in the high-impression tier. If CTR is flat or higher
in the high-impression tier, that's **OPPOSITE** or **FALSE**, and it's worth noting
plainly — a clearly-explained negative here is still useful, since it means the rule
should not over-weight raw impression volume.

## 2. Build the ranked queue (writes the CSV)

Score every `(content_hash_id, client_hash_id)` pair in March 2026 against its own
position-band benchmark CTR, rank descending, and write the queue to
`work/outputs/baseline_action_score.csv`.

In [ ]:
queue = con.sql(f"""
WITH per_page AS (
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS gsc_impressions,
           SUM(gsc_clicks) AS gsc_clicks,
           SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS gsc_avg_position,
           SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr
    FROM read_parquet('{FACT_DAILY}')
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
    GROUP BY 1,2
    HAVING SUM(gsc_impressions) > 0
),
bucketed AS (
    SELECT *,
        CASE
          WHEN gsc_avg_position <= 3 THEN '1-3'
          WHEN gsc_avg_position <= 10 THEN '4-10'
          WHEN gsc_avg_position <= 20 THEN '11-20'
          WHEN gsc_avg_position <= 50 THEN '21-50'
          ELSE '51+'
        END AS position_bucket
    FROM per_page
),
benchmarks AS (
    SELECT position_bucket,
           SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS benchmark_ctr
    FROM bucketed
    GROUP BY 1
)
SELECT
    b.content_hash_id, b.client_hash_id,
    b.gsc_impressions, b.gsc_clicks, ROUND(b.ctr, 4) AS ctr,
    b.position_bucket, ROUND(bm.benchmark_ctr, 4) AS benchmark_ctr,
    ROUND(GREATEST(bm.benchmark_ctr - b.ctr, 0) * b.gsc_impressions, 2) AS score,
    CASE WHEN bm.benchmark_ctr > b.ctr THEN 'CTR_BELOW_POSITION_BENCHMARK' ELSE 'AT_OR_ABOVE_BENCHMARK' END AS reason_code,
    CASE WHEN bm.benchmark_ctr > b.ctr THEN 'REVIEW_TITLE_AND_META' ELSE 'MONITOR' END AS action_label
FROM bucketed b
JOIN benchmarks bm USING (position_bucket)
ORDER BY score DESC
""").df()

os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(f"Wrote {len(queue)} rows to work/outputs/baseline_action_score.csv")
queue.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote 176738 rows to work/outputs/baseline_action_score.csv


,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,ctr,position_bucket,benchmark_ctr,score,reason_code,action_label
0,content_44f34c0a90047651,client_23a62021009f63c4,212404.0,24.0,0.0001,1-3,0.0039,799.37,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
1,content_8e1334d6356668e3,client_73cda7b4e4f265ea,134984.0,1.0,0.0000,1-3,0.0039,522.26,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
2,content_8d7d99f109e19aa2,client_e547b89c05043229,203497.0,289.0,0.0014,1-3,0.0039,499.84,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
3,content_fec55986a1868d62,client_73cda7b4e4f265ea,124075.0,1.0,0.0000,1-3,0.0039,479.97,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
4,content_34a70fea29d15f24,client_62f4a7e64f5e0096,143019.0,43.0,0.0003,4-10,0.0032,421.64,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
5,content_7c6373141eae744a,client_62f4a7e64f5e0096,132593.0,83.0,0.0006,4-10,0.0032,347.77,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
6,content_f6116743b00afc2d,client_62f4a7e64f5e0096,107584.0,15.0,0.0001,4-10,0.0032,334.52,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
7,content_9c057b66c30a3abb,client_73cda7b4e4f265ea,83834.0,1.0,0.0000,1-3,0.0039,323.98,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
8,content_acbcc847f8996314,client_62f4a7e64f5e0096,170808.0,262.0,0.0015,4-10,0.0032,292.92,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
9,content_cd3d932d4e1c8db0,client_9958f0a7ae1df715,89332.0,4.0,0.0000,4-10,0.0032,286.22,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META


## 3. Top-10 review

For each of the top 10 flagged rows: the action, why it's there, and what would make it
wrong.

In [ ]:
top10 = queue.head(10).reset_index(drop=True)

for i, row in top10.iterrows():
    print(f"{i+1}. content={row['content_hash_id'][:12]}... client={row['client_hash_id'][:12]}...")
    print(f"   Action: {row['action_label']} | Reason: {row['reason_code']}")
    print(f"   Why: CTR {row['ctr']:.4f} vs. benchmark {row['benchmark_ctr']:.4f} for "
          f"position band {row['position_bucket']}, on {row['gsc_impressions']:.0f} impressions "
          f"→ score {row['score']:.1f}")
    print(f"   What would make it wrong: [FILL IN — e.g. a recent title/meta change already "
          f"in progress that this month's average hasn't caught up to yet, or a low-relevance "
          f"query mix inflating impressions without real click intent]")
    print()

top10

1. content=content_44f3... client=client_23a62...
   Action: REVIEW_TITLE_AND_META | Reason: CTR_BELOW_POSITION_BENCHMARK
   Why: CTR 0.0001 vs. benchmark 0.0039 for position band 1-3, on 212404 impressions → score 799.4
   What would make it wrong: [FILL IN — e.g. a recent title/meta change already in progress that this month's average hasn't caught up to yet, or a low-relevance query mix inflating impressions without real click intent]

2. content=content_8e13... client=client_73cda...
   Action: REVIEW_TITLE_AND_META | Reason: CTR_BELOW_POSITION_BENCHMARK
   Why: CTR 0.0000 vs. benchmark 0.0039 for position band 1-3, on 134984 impressions → score 522.3
   What would make it wrong: [FILL IN — e.g. a recent title/meta change already in progress that this month's average hasn't caught up to yet, or a low-relevance query mix inflating impressions without real click intent]

3. content=content_8d7d... client=client_e547b...
   Action: REVIEW_TITLE_AND_META | Reason: CTR_BELOW_POSITION_BE

,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,ctr,position_bucket,benchmark_ctr,score,reason_code,action_label
0,content_44f34c0a90047651,client_23a62021009f63c4,212404.0,24.0,0.0001,1-3,0.0039,799.37,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
1,content_8e1334d6356668e3,client_73cda7b4e4f265ea,134984.0,1.0,0.0000,1-3,0.0039,522.26,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
2,content_8d7d99f109e19aa2,client_e547b89c05043229,203497.0,289.0,0.0014,1-3,0.0039,499.84,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
3,content_fec55986a1868d62,client_73cda7b4e4f265ea,124075.0,1.0,0.0000,1-3,0.0039,479.97,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
4,content_34a70fea29d15f24,client_62f4a7e64f5e0096,143019.0,43.0,0.0003,4-10,0.0032,421.64,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
5,content_7c6373141eae744a,client_62f4a7e64f5e0096,132593.0,83.0,0.0006,4-10,0.0032,347.77,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
6,content_f6116743b00afc2d,client_62f4a7e64f5e0096,107584.0,15.0,0.0001,4-10,0.0032,334.52,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
7,content_9c057b66c30a3abb,client_73cda7b4e4f265ea,83834.0,1.0,0.0000,1-3,0.0039,323.98,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
8,content_acbcc847f8996314,client_62f4a7e64f5e0096,170808.0,262.0,0.0015,4-10,0.0032,292.92,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
9,content_cd3d932d4e1c8db0,client_9958f0a7ae1df715,89332.0,4.0,0.0000,4-10,0.0032,286.22,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META


Fill in the bracketed "what would make it wrong" reasoning above for each of the 10 rows
after reading the printed output — this is meant to be your own skeptical read of each
specific row's numbers, not a generic caveat.

## 4. Weak picks + leakage check

Which picks look wrong, and confirmation that no future window or product flag leaked in.

In [ ]:
# Weak picks: very high score driven by very small impression counts (noisy sample)
weak_picks = queue[(queue['score'] > 0) & (queue['gsc_impressions'] < 50)].sort_values('score', ascending=False)
print(f"{len(weak_picks)} flagged rows are based on fewer than 50 impressions — likely too noisy to act on.")
weak_picks.head(10)

44242 flagged rows are based on fewer than 50 impressions — likely too noisy to act on.


,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,ctr,position_bucket,benchmark_ctr,score,reason_code,action_label
69079,content_ec099239c58580b8,client_23a62021009f63c4,49.0,0.0,0.0,1-3,0.0039,0.19,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
70000,content_08f164fd39ef4035,client_73cda7b4e4f265ea,48.0,0.0,0.0,1-3,0.0039,0.19,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
70001,content_0daed3e29d82d3fd,client_73cda7b4e4f265ea,49.0,0.0,0.0,1-3,0.0039,0.19,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
69999,content_97145417b6df70f0,client_73cda7b4e4f265ea,48.0,0.0,0.0,1-3,0.0039,0.19,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
69995,content_7a5f78b39e48bda5,client_e5c2aa26a8598242,49.0,0.0,0.0,1-3,0.0039,0.19,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
69612,content_4cb544a8ec098b9c,client_73cda7b4e4f265ea,49.0,0.0,0.0,1-3,0.0039,0.19,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
69385,content_a8bed4a7eea8ced9,client_a80fca3f171ed1de,49.0,0.0,0.0,1-3,0.0039,0.19,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
69280,content_4c9ce8cb7a233c0f,client_73cda7b4e4f265ea,48.0,0.0,0.0,1-3,0.0039,0.19,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
69294,content_110f9e9bdf53d6b5,client_a80fca3f171ed1de,48.0,0.0,0.0,1-3,0.0039,0.19,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META
69297,content_40bf56802c4b9ecc,client_a80fca3f171ed1de,49.0,0.0,0.0,1-3,0.0039,0.19,CTR_BELOW_POSITION_BENCHMARK,REVIEW_TITLE_AND_META


In [ ]:
# Leakage check: confirm only contemporaneous GSC columns were used, no future window,
# no product/label-derived flags
used_columns = ['content_hash_id', 'client_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position']
print("Columns used to build the score:", used_columns)
print("Time window used: month = '2026-03' only, no reference to any other month or future date.")
print("No pre-existing flag columns (e.g. is_fixed, refresh_flag) were joined in — score is derived only from raw GSC counts.")

Columns used to build the score: ['content_hash_id', 'client_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position']
Time window used: month = '2026-03' only, no reference to any other month or future date.
No pre-existing flag columns (e.g. is_fixed, refresh_flag) were joined in — score is derived only from raw GSC counts.


The weak picks above show why a raw score alone isn't enough: a handful of impressions can
produce a large *relative* CTR gap by chance. A production version of this rule would add a
minimum-impressions floor (e.g. `gsc_impressions >= 50`) before flagging, rather than
scoring every row regardless of sample size.

## Self-check
- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.